# Problem Statement

## **Business Context**

"Visit with Us," a leading travel company, is revolutionizing the tourism industry by leveraging data-driven strategies to optimize operations and customer engagement. While introducing a new package offering, such as the Wellness Tourism Package, the company faces challenges in targeting the right customers efficiently. The manual approach to identifying potential customers is inconsistent, time-consuming, and prone to errors, leading to missed opportunities and suboptimal campaign performance.

To address these issues, the company aims to implement a scalable and automated system that integrates customer data, predicts potential buyers, and enhances decision-making for marketing strategies. By utilizing an MLOps pipeline, the company seeks to achieve seamless integration of data preprocessing, model development, deployment, and CI/CD practices for continuous improvement. This system will ensure efficient targeting of customers, timely updates to the predictive model, and adaptation to evolving customer behaviors, ultimately driving growth and customer satisfaction.


## **Objective**

As an MLOps Engineer at "Visit with Us," your responsibility is to design and deploy an MLOps pipeline on GitHub to automate the end-to-end workflow for predicting customer purchases. The primary objective is to build a model that predicts whether a customer will purchase the newly introduced Wellness Tourism Package before contacting them. The pipeline will include data cleaning, preprocessing, transformation, model building, training, evaluation, and deployment, ensuring consistent performance and scalability. By leveraging GitHub Actions for CI/CD integration, the system will enable automated updates, streamline model deployment, and improve operational efficiency. This robust predictive solution will empower policymakers to make data-driven decisions, enhance marketing strategies, and effectively target potential customers, thereby driving customer acquisition and business growth.

## **Data Description**

The dataset contains customer and interaction data that serve as key attributes for predicting the likelihood of purchasing the Wellness Tourism Package. The detailed attributes are:

**Customer Details**
- **CustomerID:** Unique identifier for each customer.
- **ProdTaken:** Target variable indicating whether the customer has purchased a package (0: No, 1: Yes).
- **Age:** Age of the customer.
- **TypeofContact:** The method by which the customer was contacted (Company Invited or Self Inquiry).
- **CityTier:** The city category based on development, population, and living standards (Tier 1 > Tier 2 > Tier 3).
- **Occupation:** Customer's occupation (e.g., Salaried, Freelancer).
- **Gender:** Gender of the customer (Male, Female).
- **NumberOfPersonVisiting:** Total number of people accompanying the customer on the trip.
- **PreferredPropertyStar:** Preferred hotel rating by the customer.
- **MaritalStatus:** Marital status of the customer (Single, Married, Divorced).
- **NumberOfTrips:** Average number of trips the customer takes annually.
- **Passport:** Whether the customer holds a valid passport (0: No, 1: Yes).
- **OwnCar:** Whether the customer owns a car (0: No, 1: Yes).
- **NumberOfChildrenVisiting:** Number of children below age 5 accompanying the customer.
- **Designation:** Customer's designation in their current organization.
- **MonthlyIncome:** Gross monthly income of the customer.

**Customer Interaction Data**
- **PitchSatisfactionScore:** Score indicating the customer's satisfaction with the sales pitch.
- **ProductPitched:** The type of product pitched to the customer.
- **NumberOfFollowups:** Total number of follow-ups by the salesperson after the sales pitch.-
- **DurationOfPitch:** Duration of the sales pitch delivered to the customer.


# Model Building

In [1]:
# Create a master folder to keep all files created when executing the below code cells
import os
os.makedirs("TourismProject", exist_ok=True)

In [2]:
# Create a folder for storing the model building files
os.makedirs("TourismProject/models", exist_ok=True)

## Data Registration

In [3]:
os.makedirs("TourismProject/data", exist_ok=True)

Once the **data** folder created after executing the above cell, please upload the **tourism.csv** in to the folder

### Push file to hugging face account

In [6]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_file(
    path_or_fileobj="TourismProject/data/tourism.csv",
    path_in_repo="tourism.csv",
    repo_id="itisashokkumar/visit-with-us-dataset",
    repo_type="dataset",
)


No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/itisashokkumar/visit-with-us-dataset/commit/d68a191dc7594a50d55b5132c53719f7c7036757', commit_message='Upload tourism.csv with huggingface_hub', commit_description='', oid='d68a191dc7594a50d55b5132c53719f7c7036757', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/itisashokkumar/visit-with-us-dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='itisashokkumar/visit-with-us-dataset'), pr_revision=None, pr_num=None)

## Data Preparation

In [7]:
from datasets import load_dataset

dataset = load_dataset(
    "itisashokkumar/visit-with-us-dataset"
)

df = dataset["train"].to_pandas()

### Data Overview

In [8]:
print(df.shape)
print(df.head())
print(df.info())

(4128, 21)
   Unnamed: 0  CustomerID  ProdTaken   Age    TypeofContact  CityTier  \
0           0      200000          1  41.0     Self Enquiry         3   
1           1      200001          0  49.0  Company Invited         1   
2           2      200002          1  37.0     Self Enquiry         1   
3           3      200003          0  33.0  Company Invited         1   
4           5      200005          0  32.0  Company Invited         1   

   DurationOfPitch   Occupation  Gender  NumberOfPersonVisiting  ...  \
0              6.0     Salaried  Female                       3  ...   
1             14.0     Salaried    Male                       3  ...   
2              8.0  Free Lancer    Male                       3  ...   
3              9.0     Salaried  Female                       2  ...   
4              8.0     Salaried    Male                       3  ...   

   ProductPitched PreferredPropertyStar  MaritalStatus NumberOfTrips  \
0          Deluxe                   3.0      

### Data Inspection

In [9]:
print(df.isnull().sum())

print(df.duplicated().sum())

print(df.describe())

print(df.dtypes)

Unnamed: 0                  0
CustomerID                  0
ProdTaken                   0
Age                         0
TypeofContact               0
CityTier                    0
DurationOfPitch             0
Occupation                  0
Gender                      0
NumberOfPersonVisiting      0
NumberOfFollowups           0
ProductPitched              0
PreferredPropertyStar       0
MaritalStatus               0
NumberOfTrips               0
Passport                    0
PitchSatisfactionScore      0
OwnCar                      0
NumberOfChildrenVisiting    0
Designation                 0
MonthlyIncome               0
dtype: int64
0
        Unnamed: 0     CustomerID    ProdTaken          Age     CityTier  \
count  4128.000000    4128.000000  4128.000000  4128.000000  4128.000000   
mean   2527.763808  202527.763808     0.193072    37.231831     1.663275   
std    1409.439133    1409.439133     0.394757     9.174521     0.920640   
min       0.000000  200000.000000     0.000000    1

### Data Cleaning

#### Remove Unnessary columns

In [12]:
df.drop(columns=["Unnamed: 0"], inplace=True)
df.drop(columns=["CustomerID"], inplace=True)

In [14]:
print(df.shape)
print(df.isnull().sum())

(4128, 19)
ProdTaken                   0
Age                         0
TypeofContact               0
CityTier                    0
DurationOfPitch             0
Occupation                  0
Gender                      0
NumberOfPersonVisiting      0
NumberOfFollowups           0
ProductPitched              0
PreferredPropertyStar       0
MaritalStatus               0
NumberOfTrips               0
Passport                    0
PitchSatisfactionScore      0
OwnCar                      0
NumberOfChildrenVisiting    0
Designation                 0
MonthlyIncome               0
dtype: int64


No cleanup required for handling null values

### Save Cleaned Dataset

In [16]:
df.to_csv(
    "TourismProject/data/tourism_cleaned.csv",
    index=False
)

### Split Features & Target

In [17]:
X = df.drop(columns=["ProdTaken"])

y = df["ProdTaken"]

In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

#### Seperate Test & Train to Two different file

In [20]:
train = X_train.copy()
train["ProdTaken"] = y_train

test = X_test.copy()
test["ProdTaken"] = y_test

train.to_csv(
    "TourismProject/data/train.csv",
    index=False
)

test.to_csv(
    "TourismProject/data/test.csv",
    index=False
)

### Upload to huggingface

In [21]:

api.upload_file(
    path_or_fileobj="TourismProject/data/train.csv",
    path_in_repo="train.csv",
    repo_id="itisashokkumar/visit-with-us-dataset",
    repo_type="dataset",
)

api.upload_file(
    path_or_fileobj="TourismProject/data/test.csv",
    path_in_repo="test.csv",
    repo_id="itisashokkumar/visit-with-us-dataset",
    repo_type="dataset",
)

api.upload_file(
    path_or_fileobj="TourismProject/data/tourism_cleaned.csv",
    path_in_repo="tourism_cleaned.csv",
    repo_id="itisashokkumar/visit-with-us-dataset",
    repo_type="dataset",
)

CommitInfo(commit_url='https://huggingface.co/datasets/itisashokkumar/visit-with-us-dataset/commit/d697ed22c7b3a3598618a9d21a2bbe808a5ec762', commit_message='Upload tourism_cleaned.csv with huggingface_hub', commit_description='', oid='d697ed22c7b3a3598618a9d21a2bbe808a5ec762', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/itisashokkumar/visit-with-us-dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='itisashokkumar/visit-with-us-dataset'), pr_revision=None, pr_num=None)

## Model Training and Registration with Experimentation Tracking

# Deployment

## Dockerfile

In [ ]:
os.makedirs("tourism_project/deployment", exist_ok=True)

In [ ]:
%%writefile tourism_project/deployment/Dockerfile
# Use a minimal base image with Python 3.9 installed
FROM python:3.9

# Set the working directory inside the container to /app
WORKDIR /app

# Copy all files from the current directory on the host to the container's /app directory
COPY . .

# Install Python dependencies listed in requirements.txt
RUN pip3 install -r requirements.txt

RUN useradd -m -u 1000 user
USER user
ENV HOME=/home/user \
	PATH=/home/user/.local/bin:$PATH

WORKDIR $HOME/app

COPY --chown=user . $HOME/app

# Define the command to run the Streamlit app on port "8501" and make it accessible externally
CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0", "--server.enableXsrfProtection=false"]

Writing tourism_project/deployment/Dockerfile


## Streamlit App

Please ensure that the web app script is named `app.py`.

## Dependency Handling

Please ensure that the dependency handling file is named `requirements.txt`.

# Hosting

# MLOps Pipeline with Github Actions Workflow

**Note:**

1. Before running the file below, make sure to add the HF_TOKEN to your GitHub secrets to enable authentication between GitHub and Hugging Face.
2. The below code is for a sample YAML file that can be updated as required to meet the requirements of this project.

```
name: Tourism Project Pipeline

on:
  push:
    branches:
      - main  # Automatically triggers on push to the main branch

jobs:

  register-dataset:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: <add_code_here>
      - name: Upload Dataset to Hugging Face Hub
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: <add_code_here>

  data-prep:
    needs: register-dataset
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: <add_code_here>
      - name: Run Data Preparation
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: <add_code_here>


  model-traning:
    needs: data-prep
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: <add_code_here>
      - name: Start MLflow Server
        run: |
          nohup mlflow ui --host 0.0.0.0 --port 5000 &  # Run MLflow UI in the background
          sleep 5  # Wait for a moment to let the server starts
      - name: Model Building
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: <add_code_here>


  deploy-hosting:
    runs-on: ubuntu-latest
    needs: [model-traning,data-prep,register-dataset]
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: <add_code_here>
      - name: Push files to Frontend Hugging Face Space
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: <add_code_here>

```

**Note:** To use this YAML file for our use case, we need to

1. Go to the GitHub repository for the project
2. Create a folder named ***.github/workflows/***
3. In the above folder, create a file named ***pipeline.yml***
4. Copy and paste the above content for the YAML file into the ***pipeline.yml*** file

## Requirements file for the Github Actions Workflow

## Github Authentication and Push Files

* Before moving forward, we need to generate a secret token to push files directly from Colab to the GitHub repository.
* Please follow the below instructions to create the GitHub token:
    - Open your GitHub profile.
    - Click on ***Settings***.
    - Go to ***Developer Settings***.
    - Expand the ***Personal access tokens*** section and select ***Tokens (classic)***.
    - Click ***Generate new token***, then choose ***Generate new token (classic)***.
    - Add a note and select all required scopes.
    - Click ***Generate token***.
    - Copy the generated token and store it safely in a notepad.

In [ ]:
# Install Git
!apt-get install git

# Set your Git identity (replace with your details)
!git config --global user.email "<-------GitHub Email Address------->"
!git config --global user.name "<--------GitHub UserName--------->"

# Clone your GitHub repository
!git clone https://github.com/<--------GitHub UserName--------->/<--------GitHub Reponame--------->.git

# Move your folder to the repository directory
!mv /content/tourism_project/ /content/<--------GitHub Reponame--------->

In [ ]:
# Change directory to the cloned repository
%cd <--------GitHub Reponame--------->/

# Add the new folder to Git
!git add .

# Commit the changes
!git commit -m "first commit"

# Push to GitHub (you'll need your GitHub credentials; use a personal access token if 2FA enabled)
!git push https://<--------GitHub UserName--------->:<--------GitHub Token--------->@github.com/<--------GitHub UserName--------->/<--------GitHub Reponame--------->.git

# Output Evaluation

- GitHub (link to repository, screenshot of folder structure and executed workflow)

- Streamlit on Hugging Face (link to HF space, screenshot of Streamlit app)

<font size=6 color="navyblue">Power Ahead!</font>
___